# 05 - Validacao Robusta

Walk-forward, backtesting sazonal e teste de estabilidade.

**Nota:** Este notebook treina modelos temporarios para validacao. Nenhum modelo e salvo.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
from datetime import datetime

import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit

from src.features.feature_engineering import FeatureEngineer
from src.models.evaluation import ModelEvaluator

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Preparacao

In [ ]:
df = pd.read_parquet('../data/processed/processed_data.parquet')
fe = FeatureEngineer()
df_features = fe.create_all_features(df)

exclude_cols = ['timestamp', 'consumption_mw', 'region', 'holiday_name', 'city', 'date']
for col in df_features.columns:
    if col in exclude_cols:
        continue
    if pd.api.types.is_datetime64_any_dtype(df_features[col]) or df_features[col].dtype == 'object':
        exclude_cols.append(col)

features = [c for c in df_features.columns if c not in exclude_cols]

df_sorted = df_features.sort_values('timestamp').reset_index(drop=True)
X = df_sorted[features].values
y = df_sorted['consumption_mw'].values

print(f"Dados: {len(X):,} | Features: {len(features)}")

## 2. Walk-Forward Validation

In [ ]:
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
evaluator = ModelEvaluator()

fold_results = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        random_state=42, n_jobs=-1, verbosity=0
    )
    model.fit(X[train_idx], y[train_idx])
    y_pred = model.predict(X[test_idx])

    m = evaluator.calculate_metrics(y[test_idx], y_pred)
    m['fold'] = fold
    m['train_size'] = len(train_idx)
    m['test_size'] = len(test_idx)
    fold_results.append(m)

    print(f"Fold {fold}/{N_SPLITS}: MAPE={m['mape']:.2f}% R2={m['r2']:.4f} (train={len(train_idx):,} test={len(test_idx):,})")

df_folds = pd.DataFrame(fold_results)
print(f"\nMedia: MAPE={df_folds['mape'].mean():.2f}% +/- {df_folds['mape'].std():.2f}%")
print(f"       R2={df_folds['r2'].mean():.4f} +/- {df_folds['r2'].std():.4f}")

## 3. Backtesting Sazonal

In [ ]:
df_sorted['month_val'] = df_sorted['timestamp'].dt.month
df_sorted['season'] = df_sorted['month_val'].map(
    lambda m: 'Inverno' if m in [12,1,2] else 'Primavera' if m in [3,4,5] else 'Verao' if m in [6,7,8] else 'Outono'
)

season_results = []
for season in ['Primavera', 'Verao', 'Outono', 'Inverno']:
    mask = df_sorted['season'] == season
    df_season = df_sorted[mask].reset_index(drop=True)

    split = int(0.7 * len(df_season))
    X_train_s = df_season.iloc[:split][features].values
    y_train_s = df_season.iloc[:split]['consumption_mw'].values
    X_test_s = df_season.iloc[split:][features].values
    y_test_s = df_season.iloc[split:]['consumption_mw'].values

    model_s = xgb.XGBRegressor(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        random_state=42, n_jobs=-1, verbosity=0
    )
    model_s.fit(X_train_s, y_train_s)
    y_pred_s = model_s.predict(X_test_s)

    m = evaluator.calculate_metrics(y_test_s, y_pred_s)
    m['season'] = season
    m['n_samples'] = len(X_test_s)
    season_results.append(m)

    print(f"{season:12s}: MAPE={m['mape']:.2f}% R2={m['r2']:.4f} (n={len(X_test_s):,})")

df_seasons = pd.DataFrame(season_results)

## 4. Estabilidade (Random Seeds)

In [ ]:
stability_results = []
seeds = list(range(42, 52))

train_end = int(0.70 * len(df_sorted))
test_start = int(0.85 * len(df_sorted))
X_train_st = df_sorted.iloc[:train_end][features].values
y_train_st = df_sorted.iloc[:train_end]['consumption_mw'].values
X_test_st = df_sorted.iloc[test_start:][features].values
y_test_st = df_sorted.iloc[test_start:]['consumption_mw'].values

for seed in seeds:
    model_st = xgb.XGBRegressor(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        random_state=seed, n_jobs=-1, verbosity=0
    )
    model_st.fit(X_train_st, y_train_st)
    y_pred_st = model_st.predict(X_test_st)

    m = evaluator.calculate_metrics(y_test_st, y_pred_st)
    m['seed'] = seed
    stability_results.append(m)

df_stability = pd.DataFrame(stability_results)
print(f"Estabilidade ({len(seeds)} seeds):")
print(f"  MAPE: {df_stability['mape'].mean():.3f}% +/- {df_stability['mape'].std():.5f}%")
print(f"  R2:   {df_stability['r2'].mean():.5f} +/- {df_stability['r2'].std():.6f}")
print(f"  Conclusao: {'Estavel' if df_stability['mape'].std() < 0.1 else 'Instavel'}")

## 5. Visualizacao

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(range(1, N_SPLITS+1), df_folds['mape'], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Walk-Forward: MAPE por Fold')
axes[0].axhline(df_folds['mape'].mean(), color='red', linestyle='--', label=f"Media={df_folds['mape'].mean():.2f}%")
axes[0].legend()

axes[1].bar(df_seasons['season'], df_seasons['mape'], color='coral', edgecolor='black')
axes[1].set_ylabel('MAPE (%)')
axes[1].set_title('Backtesting Sazonal')

axes[2].plot(seeds, df_stability['mape'], 'g-o')
axes[2].set_xlabel('Random Seed')
axes[2].set_ylabel('MAPE (%)')
axes[2].set_title('Estabilidade por Seed')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Resumo

In [ ]:
print("=" * 70)
print("RESUMO DA VALIDACAO ROBUSTA")
print("=" * 70)
print(f"""
Walk-Forward ({N_SPLITS} folds):
  MAPE: {df_folds['mape'].mean():.2f}% +/- {df_folds['mape'].std():.2f}%
  R2:   {df_folds['r2'].mean():.4f} +/- {df_folds['r2'].std():.4f}
  Fold 1 vs Fold {N_SPLITS}: {df_folds.iloc[0]['mape']:.2f}% vs {df_folds.iloc[-1]['mape']:.2f}%

Backtesting Sazonal:
  Melhor: {df_seasons.loc[df_seasons['mape'].idxmin(), 'season']} ({df_seasons['mape'].min():.2f}%)
  Pior:   {df_seasons.loc[df_seasons['mape'].idxmax(), 'season']} ({df_seasons['mape'].max():.2f}%)
  Range:  {df_seasons['mape'].max() - df_seasons['mape'].min():.2f} pp

Estabilidade:
  MAPE std: {df_stability['mape'].std():.5f}% ({'Estavel' if df_stability['mape'].std() < 0.1 else 'Instavel'})
  R2 std:   {df_stability['r2'].std():.6f}
""")
print(f"Fim: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")